# AI-Generated Text Detection Project
MSc Data Science — Group Final Project

HC3 Dataset — Load, Flatten, Label, Clean

In [1]:
! pip install langdetect

# Imports & Constants
import os
import re
import string
import pandas as pd
from datasets import load_dataset

# langdetect lets us filter out non-English rows.
# HC3 contains a small Chinese-language subset we don't want.
from langdetect import detect, LangDetectException
RANDOM_SEED = 42

LABEL_HUMAN = 0   # human-written text
LABEL_AI    = 1   # AI-generated text (ChatGPT in HC3's case)


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# ── Dataset identifier ────────────────────────────────────────────────────────
# We tag every row with which dataset it came from.
# This column is critical later for cross-dataset evaluation —
# training on HC3 and testing on DAIGT requires knowing which is which.
DATASET_NAME = "HC3"

# ── Minimum text length ───────────────────────────────────────────────────────
# Texts shorter than this (in words) are dropped.
# A 5-word answer carries almost no stylometric signal and could be noise.
MIN_WORD_COUNT = 10
 
# ── Output path ──────────────────────────────────────────────────────────────
OUTPUT_DIR  = "/Users/yashaswini11/Desktop/Team_project/project"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "hc3_clean.csv")
 
# Create output directory if it doesn't already exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# =============================================================================
# Load Raw HC3
# =============================================================================
# HC3 lives on HuggingFace Hub. load_dataset streams it directly into memory —
# no manual download needed. The 'all' config loads all five domains together:
#   reddit_eli5, finance, medicine, open_qa, wiki_csai
#
# Raw schema of each row:
#   id              (str)  — unique question identifier
#   question        (str)  — the question asked
#   human_answers   (list) — list of human-written answers (1 or more per row)
#   chatgpt_answers (list) — list of ChatGPT-written answers (usually 1 per row)
#   source          (str)  — which of the 5 domains this question came from
# =============================================================================
 
print("=" * 60)
print("STEP 1: Loading HC3 from HuggingFace Hub")
print("=" * 60)
 
hc3_raw = load_dataset('Hello-SimpleAI/HC3', 'all', split='train')
 
print(f"  Raw rows loaded : {len(hc3_raw):,}")
print(f"  Columns         : {hc3_raw.column_names}")
 
# Quick sanity check — peek at the first row to confirm the structure
first_row = hc3_raw[0]
print(f"\n  Sample row (id={first_row['id']}):")
print(f"    source          : {first_row['source']}")
print(f"    question        : {first_row['question'][:80]}...")
print(f"    human_answers   : {len(first_row['human_answers'])} answer(s)")
print(f"    chatgpt_answers : {len(first_row['chatgpt_answers'])} answer(s)")

STEP 1: Loading HC3 from HuggingFace Hub
  Raw rows loaded : 24,322
  Columns         : ['id', 'question', 'human_answers', 'chatgpt_answers', 'source']

  Sample row (id=0):
    source          : reddit_eli5
    question        : Why is every book I hear about a " NY Times # 1 Best Seller " ? ELI5 : Why is ev...
    human_answers   : 3 answer(s)
    chatgpt_answers : 1 answer(s)


In [4]:
# =============================================================================
# SECTION 2 — Convert HuggingFace Dataset to Pandas DataFrame
# =============================================================================
# HuggingFace Dataset objects are great for streaming but Pandas DataFrames
# are much easier for the cleaning and feature engineering steps that follow.
# .to_pandas() creates a full in-memory copy - fine for HC3's 147 MB size.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 2: Converting to Pandas DataFrame")
print("=" * 60)
 
hc3_df = hc3_raw.to_pandas()
 
print(f"  DataFrame shape : {hc3_df.shape}")
print(f"  Domains present : {hc3_df['source'].unique().tolist()}")
print(f"\n  Rows per domain:")
print(hc3_df['source'].value_counts().to_string())


STEP 2: Converting to Pandas DataFrame
  DataFrame shape : (24322, 5)
  Domains present : ['reddit_eli5', 'open_qa', 'wiki_csai', 'finance', 'medicine']

  Rows per domain:
source
reddit_eli5    17112
finance         3933
medicine        1248
open_qa         1187
wiki_csai        842


In [5]:
hc3_df.head()

,id,question,human_answers,chatgpt_answers,source
0,0,"Why is every book I hear about a "" NY Times # ...","[Basically there are many categories of "" Best...",[There are many different best seller lists th...,reddit_eli5
1,1,"If salt is so bad for cars , why do we use it ...",[salt is good for not dying in car crashes and...,[Salt is used on roads to help melt ice and sn...,reddit_eli5
2,2,Why do we still have SD TV channels when HD lo...,[The way it works is that old TV stations got ...,[There are a few reasons why we still have SD ...,reddit_eli5
3,3,Why has nobody assassinated Kim Jong - un He i...,[You ca n't just go around assassinating the l...,[It is generally not acceptable or ethical to ...,reddit_eli5
4,4,How was airplane technology able to advance so...,[Wanting to kill the shit out of Germans drive...,[After the Wright Brothers made the first powe...,reddit_eli5


In [6]:
# =============================================================================
# SECTION 3 — Flatten the Nested Answer Lists
# =============================================================================
# THIS IS THE KEY TRANSFORMATION for HC3.
#
# The problem: each row stores answers as Python lists, not strings.
#   human_answers   = ["Answer text one.", "Another human answer."]
#   chatgpt_answers = ["ChatGPT's answer to this question."]
#
# What we need: one row per individual answer, with an explicit label.
#
# Strategy:
#   1. Explode human_answers  → each list item becomes its own row, label = 0
#   2. Explode chatgpt_answers → each list item becomes its own row, label = 1
#   3. Stack both exploded frames on top of each other
#
# The source column is carried through so we know which domain each answer
# came from even after the explosion.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 3: Flattening nested answer lists")
print("=" * 60)
 
# ── 3a. Explode human answers ─────────────────────────────────────────────────
# .explode('human_answers') turns:
#   row: human_answers = ["answer A", "answer B"]
# into:
#   row 1: human_answers = "answer A"
#   row 2: human_answers = "answer B"
# The question, source, and id columns are duplicated for each new row.
 
human_df = (
    hc3_df[['source', 'human_answers']]     # keep only the columns we need
    .explode('human_answers')               # one row per answer string
    .rename(columns={'human_answers': 'text'})  # standardise column name
    .assign(label=LABEL_HUMAN)             # every human answer gets label 0
)
 
print(f"  Human rows after explode  : {len(human_df):,}")
 
# ── 3b. Explode ChatGPT answers ───────────────────────────────────────────────
chatgpt_df = (
    hc3_df[['source', 'chatgpt_answers']]
    .explode('chatgpt_answers')
    .rename(columns={'chatgpt_answers': 'text'})
    .assign(label=LABEL_AI)               # every AI answer gets label 1
)
 
print(f"  AI rows after explode     : {len(chatgpt_df):,}")
 
# ── 3c. Stack both frames ─────────────────────────────────────────────────────
# pd.concat stacks them vertically (axis=0).
# ignore_index=True resets the row index so it runs 0, 1, 2, ... continuously
# rather than having duplicate index values from both frames.
 
flat_df = pd.concat([human_df, chatgpt_df], axis=0, ignore_index=True)
 
print(f"  Total rows after stacking : {len(flat_df):,}")
print(f"\n  Label distribution (before cleaning):")
print(flat_df['label'].value_counts().rename({0: 'Human (0)', 1: 'AI (1)'}).to_string())


STEP 3: Flattening nested answer lists
  Human rows after explode  : 58,546
  AI rows after explode     : 27,358
  Total rows after stacking : 85,904

  Label distribution (before cleaning):
label
Human (0)    58546
AI (1)       27358


In [7]:
# =============================================================================
# SECTION 4 — Add Metadata Columns
# =============================================================================
# We add two metadata columns that don't go into model training but are
# essential for analysis, debugging, and cross-dataset evaluation later.
#
# dataset : which original dataset this row came from ("HC3" or "DAIGT")
#           When we merge HC3 + DAIGT, this lets us split them back out
#           for cross-dataset F1 evaluation.
#
# source  : already exists from the HC3 'source' field — keeps domain info
#           (reddit_eli5, finance, medicine, open_qa, wiki_csai, or "daigt")
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 4: Adding metadata columns")
print("=" * 60)
 
flat_df['dataset'] = DATASET_NAME
 
# Reorder columns for clarity
flat_df = flat_df[['text', 'label', 'source', 'dataset']]
 
print(f"  Columns now: {flat_df.columns.tolist()}")
print(f"  Sample rows:")
print(flat_df.head(3).to_string())


STEP 4: Adding metadata columns
  Columns now: ['text', 'label', 'source', 'dataset']
  Sample rows:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       text  label       source dataset
0  Basically there are many categories of " Best Seller " . Replace " Best Seller " by something like " Oscars " and every " best seller " book is basically an " oscar - winning " book . May not have won the " Best film " , but even if you won the best director or b

In [8]:
# =============================================================================
# SECTION 5 — Text Cleaning
# =============================================================================
# Raw text from HC3 contains noise that could hurt model training:
#   - URLs (https://...) carry no linguistic signal
#   - HTML tags (<b>, &amp;) are scraping artefacts
#   - Excessive whitespace, tabs, newlines break tokenisers
#   - Leading/trailing whitespace wastes feature space
#
# We do NOT:
#   - Lowercase (RoBERTa and many embeddings are case-sensitive)
#   - Remove punctuation (punctuation density IS a feature we extract later)
#   - Stem or lemmatise (neural models handle morphology themselves)
#
# All cleaning is applied through a single function so the logic is
# transparent, testable, and reusable on the DAIGT dataset.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 5: Cleaning text")
print("=" * 60)
 
def clean_text(text: str) -> str:
    """
    Clean a single text string for AI detection modelling.
 
    Operations (in order):
        1. Cast to string — handles any NaN or non-string values safely
        2. Remove URLs
        3. Remove HTML tags
        4. Decode common HTML entities
        5. Collapse whitespace (tabs, newlines → single space)
        6. Strip leading/trailing whitespace
 
    Args:
        text: Raw input string (or any type — will be cast to str)
 
    Returns:
        Cleaned string. Empty string if input was null/empty.
    """
 
    # Step 5.1 — Cast to string
    # NaN values from pandas would cause re.sub to crash.
    # str(None) → "None", str(float('nan')) → "nan"
    # We handle those edge cases in the null-drop step below.
    text = str(text)
 
    # Step 5.2 — Remove URLs
    # Matches http/https URLs and bare www. addresses.
    # URLs in HC3 answers are usually citation artefacts, not content.
    text = re.sub(r'http\S+|www\.\S+', '', text)
 
    # Step 5.3 — Remove HTML tags
    # HC3 sources from Reddit and Wikipedia which sometimes include
    # markdown-to-HTML conversion residue like <b>, <i>, <br>, </p>
    text = re.sub(r'<[^>]+>', '', text)
 
    # Step 5.4 — Decode common HTML entities
    # &amp; → &    &lt; → <    &gt; → >    &nbsp; → space
    text = text.replace('&amp;', '&')
    text = text.replace('&lt;', '<')
    text = text.replace('&gt;', '>')
    text = text.replace('&nbsp;', ' ')
    text = text.replace('&#39;', "'")
    text = text.replace('&quot;', '"')
 
    # Step 5.5 — Collapse whitespace
    # \s+ matches any whitespace (space, tab, newline, carriage return).
    # Replaces runs of whitespace with a single space.
    text = re.sub(r'\s+', ' ', text)
 
    # Step 5.6 — Strip edges
    text = text.strip()
 
    return text
 
 
# Apply the cleaning function to every row in the text column.
# This produces a new 'text' column with cleaned values.
flat_df['text'] = flat_df['text'].apply(clean_text)
 
print(f"  Cleaning applied to {len(flat_df):,} rows")


STEP 5: Cleaning text
  Cleaning applied to 85,904 rows


In [9]:
# =============================================================================
# SECTION 6 — Drop Nulls and Empty Strings
# =============================================================================
# After cleaning, some rows may have become empty strings — for example,
# a row that contained only a URL. These carry zero information and must
# be removed before we count words or train any model.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 6: Dropping nulls and empty strings")
print("=" * 60)
 
before = len(flat_df)
 
# Drop actual NaN values in the text column
flat_df = flat_df.dropna(subset=['text'])
 
# Drop rows where text is an empty string or only whitespace
flat_df = flat_df[flat_df['text'].str.strip() != '']
 
after = len(flat_df)
print(f"  Rows before : {before:,}")
print(f"  Rows after  : {after:,}")
print(f"  Dropped     : {before - after:,}")


STEP 6: Dropping nulls and empty strings
  Rows before : 85,904
  Rows after  : 85,886
  Dropped     : 18


In [10]:
# =============================================================================
# SECTION 7 — Drop Non-English Text
# =============================================================================
# HC3 contains a Chinese-language subset under the wiki_csai domain.
# Our models are trained entirely on English text, so Chinese rows would:
#   a) Confuse TF-IDF (Chinese characters tokenise very differently)
#   b) Be incompatible with GloVe embeddings (English-only vocabulary)
#   c) Potentially skew RoBERTa (roberta-base is English-only)
#
# langdetect.detect() returns a language code: 'en', 'zh-cn', 'fr', etc.
# We keep only rows where the detected language is English ('en').
#
# Note: langdetect can occasionally misclassify very short texts.
# The MIN_WORD_COUNT filter in Section 8 mitigates this because very
# short texts are the most error-prone for language detection.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 7: Filtering non-English text")
print("=" * 60)
 
def is_english(text: str) -> bool:
    """
    Return True if langdetect identifies the text as English.
 
    Args:
        text: Input string to test.
 
    Returns:
        True if English, False for any other language or detection failure.
    """
    try:
        return detect(text) == 'en'
    except LangDetectException:
        # LangDetectException is raised if the text is too short or
        # contains only symbols/numbers and no detectable language.
        # We treat these as non-English and drop them.
        return False
 
 
before = len(flat_df)
 
english_mask = flat_df['text'].apply(is_english)
flat_df = flat_df[english_mask].reset_index(drop=True)
 
after = len(flat_df)
print(f"  Rows before language filter : {before:,}")
print(f"  Rows after language filter  : {after:,}")
print(f"  Non-English rows dropped    : {before - after:,}")


STEP 7: Filtering non-English text
  Rows before language filter : 85,886
  Rows after language filter  : 85,329
  Non-English rows dropped    : 557


In [11]:
# =============================================================================
# SECTION 8 — Drop Texts Below Minimum Word Count
# =============================================================================
# Very short answers (fewer than MIN_WORD_COUNT words) are problematic:
#   - They carry almost no stylometric signal (too few words to measure
#     sentence length variation, vocabulary diversity, etc.)
#   - They are disproportionately likely to be data artefacts (e.g., a
#     human_answers entry that was just "[deleted]" or "N/A")
#   - Language detection is less reliable on short texts
#
# We count words by splitting on whitespace — fast and sufficient here.
# (We use a proper tokeniser later in feature engineering for model input.)
# =============================================================================

print("\n" + "=" * 60)
print("STEP 8: Removing texts shorter than minimum word count")
print("=" * 60)
 
# Create a word count column. We keep this permanently — it becomes a
# stylometric feature in the feature engineering notebook.
flat_df['word_count'] = flat_df['text'].str.split().str.len()
 
before = len(flat_df)
flat_df = flat_df[flat_df['word_count'] >= MIN_WORD_COUNT].reset_index(drop=True)
after = len(flat_df)

print(f"  Minimum word count threshold : {MIN_WORD_COUNT}")
print(f"  Rows before                  : {before:,}")
print(f"  Rows after                   : {after:,}")
print(f"  Dropped                      : {before - after:,}")
print(f"\n  Word count stats:")
print(flat_df['word_count'].describe().round(1).to_string())
 


STEP 8: Removing texts shorter than minimum word count
  Minimum word count threshold : 10
  Rows before                  : 85,329
  Rows after                   : 84,725
  Dropped                      : 604

  Word count stats:
count    84725.0
mean       147.3
std        141.1
min         10.0
25%         58.0
50%        120.0
75%        194.0
max       7903.0


In [12]:
# =============================================================================
# SECTION 9 — Remove Duplicate Texts
# =============================================================================
# Duplicate texts cause data leakage between train and test splits.
# If the same text appears in both train and test, the model has effectively
# already "seen" the answer during training — inflating test metrics.
#
# We deduplicate on the cleaned text string itself, keeping the first
# occurrence. We check duplicates before splitting so no text can appear
# in both train and test sets.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 9: Removing duplicate texts")
print("=" * 60)
 
before = len(flat_df)
flat_df = flat_df.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)
after = len(flat_df)
 
print(f"  Rows before deduplication : {before:,}")
print(f"  Rows after deduplication  : {after:,}")
print(f"  Duplicates removed        : {before - after:,}")



STEP 9: Removing duplicate texts
  Rows before deduplication : 84,725
  Rows after deduplication  : 78,733
  Duplicates removed        : 5,992


In [13]:
print(flat_df.columns.tolist())

['text', 'label', 'source', 'dataset', 'word_count']


In [14]:
# =============================================================================
# SECTION 10 — Schema Assertions (Data Quality Checks)
# =============================================================================
# These assertions are our automated data quality gate. If any of them
# fail, the script raises an AssertionError immediately rather than
# producing a quietly broken CSV that causes mysterious model failures later.
#
# Think of these as the unit tests for the data pipeline.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 10: Running data quality assertions")
print("=" * 60)
 
# Check 1 — Required columns all present
required_cols = {'text', 'label', 'source', 'dataset', 'word_count'}
assert required_cols.issubset(set(flat_df.columns)), \
    f"Missing columns: {required_cols - set(flat_df.columns)}"
print("  ✓ All required columns present")
 
# Check 2 — No nulls in critical columns
null_counts = flat_df[['text', 'label']].isnull().sum()
assert null_counts.sum() == 0, \
    f"Null values found:\n{null_counts}"
print("  ✓ No null values in text or label columns")
 
# Check 3 — Labels are only 0 or 1
assert flat_df['label'].isin([0, 1]).all(), \
    "Label column contains values other than 0 and 1"
print("  ✓ All labels are 0 or 1")
 
# Check 4 — Both classes present
assert flat_df['label'].nunique() == 2, \
    "Dataset contains only one class — check flattening logic"
print("  ✓ Both classes (0=Human, 1=AI) present")
 
# Check 5 — No empty strings
assert (flat_df['text'].str.strip() != '').all(), \
    "Empty string found in text column after cleaning"
print("  ✓ No empty strings in text column")
 
# Check 6 — All texts meet minimum word count
assert (flat_df['word_count'] >= MIN_WORD_COUNT).all(), \
    f"Text below minimum word count ({MIN_WORD_COUNT}) found"
print(f"  ✓ All texts have >= {MIN_WORD_COUNT} words")
 
# Check 7 — Dataset column is correct
assert (flat_df['dataset'] == DATASET_NAME).all(), \
    "Dataset column contains unexpected values"
print(f"  ✓ Dataset column correctly set to '{DATASET_NAME}'")
 
# Check 8 — No duplicate texts remain
assert flat_df['text'].duplicated().sum() == 0, \
    "Duplicate texts remain after deduplication step"
print("  ✓ No duplicate texts")
 
print("\n  ALL ASSERTIONS PASSED ✓")


STEP 10: Running data quality assertions
  ✓ All required columns present
  ✓ No null values in text or label columns
  ✓ All labels are 0 or 1
  ✓ Both classes (0=Human, 1=AI) present
  ✓ No empty strings in text column
  ✓ All texts have >= 10 words
  ✓ Dataset column correctly set to 'HC3'
  ✓ No duplicate texts

  ALL ASSERTIONS PASSED ✓


In [15]:
# =============================================================================
# SECTION 11 — Final Summary and Save
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 11: Final summary")
print("=" * 60)
 
print(f"\n  Final dataset shape : {flat_df.shape}")
 
print(f"\n  Label distribution:")
label_counts = flat_df['label'].value_counts().sort_index()
for label, count in label_counts.items():
    name = "Human" if label == 0 else "AI"
    pct  = count / len(flat_df) * 100
    print(f"    {label} ({name}) : {count:,} rows ({pct:.1f}%)")
 
print(f"\n  Rows per domain (source):")
print(flat_df['source'].value_counts().to_string())
 
print(f"\n  Sample cleaned rows:")
print(flat_df[['text', 'label', 'source']].sample(3, random_state=RANDOM_SEED).to_string())
 
# ── Save to CSV ───────────────────────────────────────────────────────────────
# index=False prevents pandas from writing the row numbers as a column.
# utf-8-sig encoding ensures Excel opens the file correctly on Windows.
flat_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
 
print(f"\n  Saved to : {OUTPUT_FILE}")
print("\n" + "=" * 60)
print("HC3 data engineering complete.")


STEP 11: Final summary

  Final dataset shape : (78733, 5)

  Label distribution:
    0 (Human) : 52,525 rows (66.7%)
    1 (AI) : 26,208 rows (33.3%)

  Rows per domain (source):
source
reddit_eli5    61572
finance         8368
open_qa         4639
medicine        2542
wiki_csai       1612

  Sample cleaned rows:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           